# Local backups

`device.backup_to_storage(root=...)` writes the backups of a device to a local folder, with the same layout as in S3: `devices/<id>/data/*.parquet` and `devices/<id>/metadata.json`. `device.load_from_storage(root=...)` reads them back, also when later backups have more channels.

To try it from a checkout of the repository:

```
gh pr checkout 72                    # or git checkout feature/local-backups
pip install -e '.[storage]'
pytest tests/unit/test_long_processing.py
```

No S3 bucket or AWS credentials are needed for this notebook.

In [1]:
import logging
import os
import shutil
from os.path import join
from unittest import mock

import pandas as pd
import scdata as sc
from scdata._config import config
from scdata.tools.custom_logger import set_logger_level

# Only errors: 'WARNING', 'INFO' or 'DEBUG' show what scdata and smartcitizen-connector do
set_logger_level('ERROR')
logging.getLogger('smartcitizen_connector').setLevel('ERROR')

sc.__version__

Found Environment file at: /home/oscar/.cache/scdata/.env


'1.8.0'

In [2]:
# A folder only for this notebook: it starts empty
root = join(config.paths['data'], 'local_backups_example')
shutil.rmtree(root, ignore_errors=True)

def tree():
    for folder, _, files in sorted(os.walk(root)):
        for file in sorted(files):
            print(os.path.relpath(join(folder, file), root))

## Some data to back up

Three days of a device from the Smart Citizen API, split in three parts.

In [3]:
device = sc.Device(params=sc.APIParams(id=18154))
device.options.min_date = (pd.Timestamp.now(tz='UTC') - pd.Timedelta('3D')).strftime('%Y-%m-%d')
await device.load()

days = [day for _, day in device.data.groupby(device.data.index.date)][-3:]
[len(day) for day in days]

[1439, 1438, 526]

## Overwrite and append

`mode='overwrite'` replaces all the backups of the device, `mode='append'` (the default) adds one more file.

In [4]:
device.data = days[0]
device.backup_to_storage(mode='overwrite', root=root)
tree()

devices/18154/metadata.json
devices/18154/data/b7747105c31e47358924d4adfe331806.parquet


In [5]:
device.data = days[1]
device.backup_to_storage(mode='append', root=root)
tree()

devices/18154/metadata.json
devices/18154/data/ac38e6e65747416782726671d3686647.parquet
devices/18154/data/b7747105c31e47358924d4adfe331806.parquet


In [6]:
stored = sc.Device(params=sc.APIParams(id=device.id))
stored.load_from_storage(root=root)
stored.data.index.min(), stored.data.index.max(), len(stored.data) == len(days[0]) + len(days[1])

(Timestamp('2026-11-16 00:00:18+0100', tz='Europe/Stockholm'),
 Timestamp('2026-11-17 23:59:50+0100', tz='Europe/Stockholm'),
 True)

## A channel added later

Each backup has the channels of that moment. The channels of all of them are read: a new one is empty before it was added.

In [7]:
device.data = days[2].assign(NEW=1.0)
device.backup_to_storage(root=root)

stored.load_from_storage(root=root)
stored.data['NEW'].isna().groupby(stored.data.index.date).mean()

2026-11-16    1.0
2026-11-17    1.0
2026-11-18    0.0
Name: NEW, dtype: float64

Only a period and some channels can be read (the filter goes to the parquet files):

In [8]:
stored.load_from_storage(root=root, min_date=str(days[1].index[0].date()), channels=['TEMP', 'NEW'])
stored.data.index.min(), list(stored.data.columns)

(Timestamp('2026-11-17 01:00:46+0100', tz='Europe/Stockholm'), ['TEMP', 'NEW'])

## A failed backup keeps the previous one

The data and `metadata.json` are written aside first. If something fails (the disk is full, `metadata.json` cannot be replaced...), the backups stay as they were, without partial files.

In [9]:
stored.load_from_storage(root=root)
before = len(stored.data)

def full(*args, **kwargs):
    raise OSError('No space left on device')

device.data = days[0]
for mode in ['overwrite', 'append']:
    with mock.patch.object(pd.DataFrame, 'to_parquet', full):
        try:
            device.backup_to_storage(mode=mode, root=root)
        except OSError as error:
            print(mode, 'failed:', error)

stored.load_from_storage(root=root)
len(stored.data) == before

overwrite failed: No space left on device
append failed: No space left on device


True

In [10]:
# The data is in place, but metadata.json cannot be replaced: the data is put back
real_replace = os.replace
def replace(source, destination):
    if destination.endswith('metadata.json'):
        raise OSError('Permission denied')
    real_replace(source, destination)

for mode in ['overwrite', 'append']:
    with mock.patch('os.replace', replace):
        try:
            device.backup_to_storage(mode=mode, root=root)
        except OSError as error:
            print(mode, 'failed:', error)

stored.load_from_storage(root=root)
print(len(stored.data) == before)
tree()

overwrite failed: Permission denied
append failed: Permission denied
True
devices/18154/metadata.json
devices/18154/data/20b8b86e518649a3b33a8916eaba08c8.parquet
devices/18154/data/ac38e6e65747416782726671d3686647.parquet
devices/18154/data/b7747105c31e47358924d4adfe331806.parquet


## Only in S3

qc data and metrics are only backed up to S3: with a local `root`, it returns `False`.

In [11]:
device.backup_to_storage(root=root, add_qc_data=True)

[2026-11-18 08:51:29,098] - scdata - ERROR - qc data and metrics are only backed up to S3 (device.py:1260)


False

In [12]:
shutil.rmtree(root)